# Reproduksi Model Penyisihan — Tim datascape
## PeDaS 2026 | Babak Final

**Notebook ini melakukan inference-only (Run All). Tidak ada proses training ulang.**

**Spesifikasi:**
- **Input:** `predict_ready.csv` (1.500 baris data uji siap pakai)
- **Model:** 20 model tersimpan (5 folds x 4 branches: XGBoost, LightGBM, CatBoost, Calibrated LinearSVC NLP)
- **Repository Mirror:** https://github.com/ZakyFauzi/pedas2026-reproduksi-penyisihan
- **Ukuran Paket ZIP:** < 1 MB (100% mematuhi batas maksimal 10 MB juknis). Bobot 20 model tersimpan diunduh secara otomatis dan paralel pada saat Run All melalui Fastly CDN GitHub mirror.
- **Ensemble:** Optimal Nelder-Mead quad-branch weighting + Powell threshold calibration + domain threat routing
- **Output:** `submission_datascape_ReproduksiPenyisihan.csv` & `submission_output.csv` (1.500 baris format `id,category`)

---


In [1]:
# CELL 1: Environment & Dependencies
# Jalankan perintah di bawah jika dependencies belum terinstal:
# !pip install xgboost lightgbm catboost scikit-learn scipy pandas numpy tqdm joblib 

In [2]:
# CELL 2: Configuration, Environment Verification & Automatic Weight Retrieval
import os, sys, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

DATA_PATH       = os.path.join(NOTEBOOK_DIR, 'predict_ready.csv')
MODEL_DIR       = os.path.join(NOTEBOOK_DIR, 'models')
WEIGHTS_DIR     = os.path.join(NOTEBOOK_DIR, 'weights')
OUTPUT_PATH     = os.path.join(NOTEBOOK_DIR, 'submission_datascape_ReproduksiPenyisihan.csv')
OUTPUT_ALT_PATH = os.path.join(NOTEBOOK_DIR, 'submission_output.csv')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'[CONFIG] NOTEBOOK_DIR : {NOTEBOOK_DIR}')
print(f'[CONFIG] DATA_PATH    : {DATA_PATH}')
print(f'[CONFIG] MODEL_DIR    : {MODEL_DIR}')
print(f'[CONFIG] WEIGHTS_DIR  : {WEIGHTS_DIR}')
print(f'[CONFIG] OUTPUT_PATH  : {OUTPUT_PATH}')

# -------------------------------------------------------------------------
# AUTO-DOWNLOAD BOBOT MODEL DARI GITHUB MIRROR
# Batasan unggah form panitia maksimal 10 MB. Bobot model (~83 MB total)
# di-host pada mirror resmi: https://github.com/ZakyFauzi/pedas2026-reproduksi-penyisihan
# Sel ini mengunduh otomatis secara paralel jika file belum ada di lokal.
# -------------------------------------------------------------------------
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/ZakyFauzi/pedas2026-reproduksi-penyisihan/main'

REQUIRED_MODELS = [
    'le_label.pkl',
    'xgb_fold0.pkl', 'xgb_fold1.pkl', 'xgb_fold2.pkl', 'xgb_fold3.pkl', 'xgb_fold4.pkl',
    'lgb_fold0.pkl', 'lgb_fold1.pkl', 'lgb_fold2.pkl', 'lgb_fold3.pkl', 'lgb_fold4.pkl',
    'cat_fold0.pkl', 'cat_fold1.pkl', 'cat_fold2.pkl', 'cat_fold3.pkl', 'cat_fold4.pkl',
    'nlp_fold0.pkl', 'nlp_fold1.pkl', 'nlp_fold2.pkl', 'nlp_fold3.pkl', 'nlp_fold4.pkl',
]

REQUIRED_WEIGHTS = [
    'opt_weights.npy',
    'best_alphas.npy'
]

downloads_needed = []

for m in REQUIRED_MODELS:
    target_file = os.path.join(MODEL_DIR, m)
    min_sz = 300 if m == 'le_label.pkl' else 1000
    if not os.path.exists(target_file) or os.path.getsize(target_file) < min_sz:
        downloads_needed.append((f'{GITHUB_RAW_BASE}/models/{m}', target_file, f'models/{m}'))

for w in REQUIRED_WEIGHTS:
    target_file = os.path.join(WEIGHTS_DIR, w)
    if not os.path.exists(target_file) or os.path.getsize(target_file) < 50:
        downloads_needed.append((f'{GITHUB_RAW_BASE}/weights/{w}', target_file, f'weights/{w}'))

if not os.path.exists(DATA_PATH) or os.path.getsize(DATA_PATH) < 1000:
    downloads_needed.append((f'{GITHUB_RAW_BASE}/predict_ready.csv', DATA_PATH, 'predict_ready.csv'))

def download_single_artifact(item):
    url, dest, label = item
    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=30) as response:
                content = response.read()
                with open(dest, 'wb') as f_out:
                    f_out.write(content)
            size_kb = os.path.getsize(dest) / 1024
            print(f'  [DOWNLOADED] {label:22s} ({size_kb:7.1f} KB) [OK]')
            return True
        except Exception as e:
            if attempt == max_retries:
                print(f'  [ERROR] Gagal mengunduh {label} setelah {max_retries}x: {e}')
                raise e
            time.sleep(1.5 * attempt)
    return False

if downloads_needed:
    print(f'\n[*] Mengunduh {len(downloads_needed)} berkas model/artefak dari mirror GitHub...')
    print('    URL Mirror : https://github.com/ZakyFauzi/pedas2026-reproduksi-penyisihan')
    start_t = time.time()
    with ThreadPoolExecutor(max_workers=5) as executor:
        list(executor.map(download_single_artifact, downloads_needed))
    elapsed = time.time() - start_t
    print(f'[OK] Unduhan selesai dalam {elapsed:.2f} detik.')
else:
    print('[OK] Semua berkas model & bobot telah tersedia secara lokal.')

print('\n[VERIFIKASI INTEGRITAS]')
for path, name in [(DATA_PATH, 'predict_ready.csv'), (MODEL_DIR, 'models/'), (WEIGHTS_DIR, 'weights/')]:
    exists = os.path.exists(path)
    status = '[OK]' if exists else '[MISSING]'
    print(f'  {status} {name}')
    if not exists:
        raise FileNotFoundError(f'Required path not found: {path}')

missing_models = [m for m in REQUIRED_MODELS if not os.path.exists(os.path.join(MODEL_DIR, m)) or os.path.getsize(os.path.join(MODEL_DIR, m)) < 100]
if missing_models:
    raise RuntimeError(f'Model belum lengkap: {missing_models}')
print(f'  [OK] Seluruh {len(REQUIRED_MODELS)} berkas model lengkap terverifikasi.')


[CONFIG] NOTEBOOK_DIR : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan
[CONFIG] DATA_PATH    : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\predict_ready.csv
[CONFIG] MODEL_DIR    : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\models
[CONFIG] WEIGHTS_DIR  : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\weights
[CONFIG] OUTPUT_PATH  : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\submission_datascape_ReproduksiPenyisihan.csv
[OK] Semua berkas model & bobot telah tersedia secara lokal.

[VERIFIKASI INTEGRITAS]
  [OK] predict_ready.csv
  [OK] models/
  [OK] weights/
  [OK] Seluruh 21 berkas model lengkap terverifikasi.


In [3]:
# CELL 3: Core Library Imports
import sys, os, re, math, warnings
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')
print('[OK] Core libraries imported successfully.')

[OK] Core libraries imported successfully.


In [4]:
# CELL 4: Load Saved Artifacts (Label Encoder, 20 Models, Weights & Alphas)

# 1. Label Encoder
le = joblib.load(os.path.join(MODEL_DIR, 'le_label.pkl'))
classes = list(le.classes_)
cls_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_cls = {i: c for i, c in enumerate(classes)}
print(f'[OK] Target classes ({len(classes)}): {classes}')

# 2. Load 5 folds x 4 branches
xgb_models = []
lgb_models = []
cat_models = []
nlp_models = []

for fold in range(5):
    xgb_models.append(joblib.load(os.path.join(MODEL_DIR, f'xgb_fold{fold}.pkl')))
    lgb_models.append(joblib.load(os.path.join(MODEL_DIR, f'lgb_fold{fold}.pkl')))
    cat_models.append(joblib.load(os.path.join(MODEL_DIR, f'cat_fold{fold}.pkl')))
    nlp_models.append(joblib.load(os.path.join(MODEL_DIR, f'nlp_fold{fold}.pkl'))) # (tfidf_pipeline, calibrated_model)

print(f'[OK] Successfully loaded 20 base models (5 folds x 4 branches).')

# 3. Load Ensemble Weights & Decision Threshold Alphas
opt_weights = np.load(os.path.join(WEIGHTS_DIR, 'opt_weights.npy'))
best_alphas = np.load(os.path.join(WEIGHTS_DIR, 'best_alphas.npy'))
print(f'[OK] Blending weights (LGB, CAT, XGB, NLP): {opt_weights}')
print(f'[OK] Calibrated threshold alphas: {best_alphas}')

[OK] Target classes (7): ['brand', 'fakeshop', 'malware', 'online gambling', 'other', 'phishing', 'spam']


[OK] Successfully loaded 20 base models (5 folds x 4 branches).
[OK] Blending weights (LGB, CAT, XGB, NLP): [0.2874 0.2734 0.2734 0.1658]
[OK] Calibrated threshold alphas: [1.55499302 1.41373397 0.80544656 1.34517202 0.51312876 1.97632732
 0.41810248]


In [5]:
# CELL 5: Ingest Preprocessed Input Data
test_df = pd.read_csv(DATA_PATH)
print(f'[OK] Input test data loaded: {test_df.shape[0]} rows, {test_df.shape[1]} columns.')
test_df.head(3)

[OK] Input test data loaded: 1500 rows, 10 columns.


,id,url,brand,discovered,confidence_level,ip,domain,sld,registrar,registration_date
0,PEDAS-bf8bc60352a3,https://claimgiveaway16jt.*******.biz.id/,Generic/Spear Phishing,4/13/2024 13:46,100,NaN,*******.biz.id,biz.id,PT Jagoan Hosting Indonesia,7/2/2026
1,PEDAS-7dd13e6859b0,https://baitulmal.************.go.id/?MENYALAA...,judi online,8/5/2024 9:55,100,123.108.101.18,************.go.id,go.id,Kementerian Komunikasi dan Informatika,3/5/2008
2,PEDAS-0fe0d926e0d2,https://fresp17icxbfrx.****.biz.id/,Garena,2/13/2024 22:45,100,104.21.43.87,****.biz.id,biz.id,PT JC Indonesia,8/20/2026


In [6]:
# CELL 6: Feature Engineering Pipeline (Deterministic, Identical to Training)
GAMBLING_KEYWORDS = [
    'slot', 'togel', 'judi', 'gacor', 'zeus', 'maxwin', 'scatter', 'jackpot', 'mahjong', 'olympus',
    'pragmatic', 'sbobet', 'habanero', 'mpo', 'wso', 'idn', 'poker', 'casino', 'toto', 'sultan',
    'spaceman', 'bet', 'cuan', 'bo', 'spbo', 'kijangwin', 'situs', 'terpercaya', 'agen', 'link',
    'daftar', 'bonus', 'depo', 'wd', 'pulsa', 'dana', 'hoki', 'menang', 'jp', 'rtp',
    'sensational', 'bocoran', 'bandar', 'taruhan', 'livecasino', 'bola', 'parlay', 'roulette',
    'baccarat', 'sicbo', 'dragontiger', 'tembakikan', 'arcade', 'pgsoft', 'joker', 'microgaming',
    'cq9', 'spadegaming', 'playtech', 'live22', 'flowgaming', 'yggdrasil', 'onegame', 'redtiger',
    'cartel', 'cuaks', 'uhuy', 'bo177', 'slot5000', 'rajawd', 'qqstar', 'member', 'jackpot88',
    'vip', 'win', 'menang88', 'hoki88', 'cuan88', 'gacor88', 'zeus88', 'slot88'
]
PHISHING_KEYWORDS = [
    'login', 'verify', 'confirm', 'akun', 'klaim', 'pulsa', 'dana', 'bantuan', 'bansos',
    'whatsapp', 'telegram', 'facebook', 'otp', 'password', 'secure', 'update', 'checkpoint',
    'comunitystandards', 'checkpoint.php', 'service', 'support', 'auth', 'security',
    'account', 'signin', 'recovery', 'helpdesk', 'meta', 'dhl', 'bri', 'bca', 'bni', 'mandiri',
    'kemensos', 'blt', 'kuota', 'undian', 'hadiah', 'event', 'cek-bansos', 'penerima'
]
MALWARE_KEYWORDS = [
    'repository', 'pustaka', 'perpustakaan', 'digilib', 'slims', 'download', 'install',
    'apk', 'exe', 'zip', 'shell', 'phpunit', 'xcc', 'payload', 'acme-challenge',
    'well-known', 'ckfinder', 'osminogs', 'exploit', 'backdoor', 'c99', 'r57',
    'alfa_data', 'notarizeds.php', 'magnitudesg.php', 'userfiles', 'uploader'
]
SHOPPING_KEYWORDS = [
    'shop', 'store', 'toko', 'olshop', 'cart', 'diskon', 'promo', 'murah', 'produk',
    'beli', 'jual', 'katalog', 'ecommerce', 'checkout', 'pesan', 'order'
]
SPAM_KEYWORDS = [
    'journey-to-stardom', 'user-agent', 'mozilla', 'applewebkit', 'apl', 'notarizeds',
    'magnitudesg', 'watermark', 'thai', 'lang/s/id', 'redirect', 'shortlink'
]
PHISHING_BRANDS = {
    'DANA', 'Facebook', 'Facebook, Inc.', 'Telegram', 'WhatsApp', 'Bank BRI', 'BRI',
    'Bank Permata', 'Coda Payments', 'Tencent', 'Garena', 'DHL', 'microsoft',
    'Generic/Spear Phishing', 'Instagram', 'Meta', 'TikTok', 'Apple', 'BCA', 'BNI'
}

def calculate_entropy(s):
    if not s: return 0.0
    prob = [float(s.count(c)) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in prob if p > 0)

def extract_features(df):
    feat_dict = {}
    urls = df['url'].fillna('').astype(str)
    full_url_lower = urls.str.lower()

    feat_dict['url_length'] = urls.str.len()
    feat_dict['url_has_https'] = urls.str.startswith('https://').astype(int)
    feat_dict['url_has_http'] = urls.str.startswith('http://').astype(int)
    feat_dict['url_no_scheme'] = (~(urls.str.startswith('http://') | urls.str.startswith('https://'))).astype(int)
    feat_dict['url_dot_count'] = urls.str.count(r'\.')
    feat_dict['url_slash_count'] = urls.str.count('/')
    feat_dict['url_hyphen_count'] = urls.str.count('-')
    feat_dict['url_underscore_count'] = urls.str.count('_')
    feat_dict['url_at_count'] = urls.str.count('@')
    feat_dict['url_question_count'] = urls.str.count(r'\?')
    feat_dict['url_equal_count'] = urls.str.count('=')
    feat_dict['url_percent_count'] = urls.str.count('%')
    feat_dict['url_digit_count'] = urls.apply(lambda u: sum(c.isdigit() for c in u))
    feat_dict['url_digit_ratio'] = feat_dict['url_digit_count'] / (feat_dict['url_length'] + 1)
    feat_dict['url_entropy'] = urls.apply(calculate_entropy)

    def parse_comp(u):
        clean = u.replace('http://', '').replace('https://', '')
        parts = clean.split('/', 1)
        domain_part = parts[0]
        path = parts[1] if len(parts) > 1 else ''
        query = path.split('?', 1)[1] if '?' in path else ''
        path_only = path.split('?', 1)[0] if '?' in path else path
        has_ext = int(bool(re.search(r'\.(php|html|htm|exe|apk|zip|js|css)$', path_only, re.I)))
        is_bare = int(len(path_only.strip('/')) == 0)
        subdomains = domain_part.count('.') - 1
        return len(domain_part), len(path), len(query), is_bare, has_ext, subdomains

    comp = urls.apply(parse_comp)
    feat_dict['domain_part_len'] = comp.apply(lambda x: x[0])
    feat_dict['path_len'] = comp.apply(lambda x: x[1])
    feat_dict['query_len'] = comp.apply(lambda x: x[2])
    feat_dict['is_root_domain'] = comp.apply(lambda x: x[3])
    feat_dict['has_file_extension'] = comp.apply(lambda x: x[4])
    feat_dict['subdomain_count'] = comp.apply(lambda x: x[5])

    feat_dict['ext_php'] = full_url_lower.str.contains(r'\.php', regex=True).astype(int)
    feat_dict['ext_exe'] = full_url_lower.str.contains(r'\.exe', regex=True).astype(int)
    feat_dict['ext_apk'] = full_url_lower.str.contains(r'\.apk', regex=True).astype(int)
    feat_dict['ext_zip'] = full_url_lower.str.contains(r'\.zip', regex=True).astype(int)
    feat_dict['ext_repo'] = full_url_lower.str.contains(r'/repository', regex=True).astype(int)
    feat_dict['path_slims'] = full_url_lower.str.contains(r'slims|pustaka|perpustakaan|digilib', regex=True).astype(int)
    feat_dict['path_wp'] = full_url_lower.str.contains(r'wp-content|wp-admin|themes', regex=True).astype(int)
    feat_dict['path_acme'] = full_url_lower.str.contains(r'acme-challenge|\.well-known', regex=True).astype(int)
    feat_dict['path_ckfinder'] = full_url_lower.str.contains(r'ckfinder|userfiles', regex=True).astype(int)

    for kw in GAMBLING_KEYWORDS:
        feat_dict[f'kw_g_{kw}'] = full_url_lower.str.contains(kw, regex=False).astype(int)
    feat_dict['kw_gambling_total'] = sum(feat_dict[f'kw_g_{kw}'] for kw in GAMBLING_KEYWORDS)
    for kw in PHISHING_KEYWORDS:
        feat_dict[f'kw_p_{kw}'] = full_url_lower.str.contains(kw, regex=False).astype(int)
    feat_dict['kw_phishing_total'] = sum(feat_dict[f'kw_p_{kw}'] for kw in PHISHING_KEYWORDS)
    for kw in MALWARE_KEYWORDS:
        feat_dict[f'kw_m_{kw}'] = full_url_lower.str.contains(kw, regex=False).astype(int)
    feat_dict['kw_malware_total'] = sum(feat_dict[f'kw_m_{kw}'] for kw in MALWARE_KEYWORDS)
    for kw in SHOPPING_KEYWORDS:
        feat_dict[f'kw_s_{kw}'] = full_url_lower.str.contains(kw, regex=False).astype(int)
    feat_dict['kw_shop_total'] = sum(feat_dict[f'kw_s_{kw}'] for kw in SHOPPING_KEYWORDS)
    for kw in SPAM_KEYWORDS:
        feat_dict[f'kw_spam_{kw}'] = full_url_lower.str.contains(kw, regex=False).astype(int)
    feat_dict['kw_spam_total'] = sum(feat_dict[f'kw_spam_{kw}'] for kw in SPAM_KEYWORDS)

    brands = df['brand'].fillna('').astype(str).str.strip()
    feat_dict['brand_is_missing'] = brands.apply(lambda b: int(b in ('', '-', 'nan', 'Other', 'other'))).astype(int)
    feat_dict['brand_is_judi'] = brands.str.lower().str.contains('judi', na=False).astype(int)
    feat_dict['brand_is_phishing_target'] = brands.apply(lambda b: int(b in PHISHING_BRANDS)).astype(int)
    feat_dict['brand_is_dana'] = (brands == 'DANA').astype(int)
    feat_dict['brand_is_fb'] = brands.str.contains('Facebook', na=False).astype(int)
    feat_dict['brand_is_wa'] = (brands == 'WhatsApp').astype(int)
    feat_dict['brand_is_tg'] = (brands == 'Telegram').astype(int)

    conf = pd.to_numeric(df['confidence_level'], errors='coerce').fillna(0)
    feat_dict['conf_raw'] = conf
    feat_dict['conf_is_90'] = (conf == 90).astype(int)
    feat_dict['conf_is_100'] = (conf == 100).astype(int)
    feat_dict['conf_is_50'] = (conf == 50).astype(int)
    feat_dict['conf_is_other'] = (~conf.isin([90, 100, 50])).astype(int)

    ips = df['ip'].fillna('').astype(str).str.strip()
    feat_dict['ip_is_missing'] = (ips.isin(['', 'nan', '0.0.0.0'])).astype(int)

    def extract_net(ip_str):
        if not ip_str or ip_str in ('nan', '0.0.0.0'):
            return 0, 0, 0
        parts = ip_str.split('.')
        if len(parts) >= 3:
            try: return int(parts[0]), int(parts[1]), int(parts[2])
            except: return 0, 0, 0
        return 0, 0, 0

    nets = ips.apply(extract_net)
    feat_dict['net_o1'] = nets.apply(lambda x: x[0])
    feat_dict['net_o2'] = nets.apply(lambda x: x[1])
    feat_dict['net_o3'] = nets.apply(lambda x: x[2])
    feat_dict['is_cloudflare'] = ((feat_dict['net_o1'] == 104) & (feat_dict['net_o2'] == 21) |
                                  (feat_dict['net_o1'] == 172) & (feat_dict['net_o2'] == 67)).astype(int)
    feat_dict['is_id_subnet'] = (feat_dict['net_o1'] == 103).astype(int)
    feat_dict['is_aws_subnet'] = (feat_dict['net_o1'].isin([52, 54, 3, 15, 18, 99])).astype(int)
    feat_dict['is_hostinger'] = (feat_dict['net_o1'] == 153).astype(int)

    slds = df['sld'].fillna('').astype(str)
    for sld_name in ['go.id', 'my.id', 'biz.id', 'ac.id', 'sch.id', 'co.id', 'id', 'desa.id', 'net.id', 'ponpes.id', 'or.id', 'mil.id']:
        feat_dict['sld_' + sld_name.replace('.', '_')] = (slds == sld_name).astype(int)

    regs = df['registrar'].fillna('').astype(str)
    feat_dict['reg_kominfo'] = regs.str.contains('Komunikasi', na=False).astype(int)
    feat_dict['reg_jc'] = regs.str.contains('JC Indonesia', na=False).astype(int)
    feat_dict['reg_jagat'] = regs.str.contains('Jagat', na=False).astype(int)
    feat_dict['reg_digital_registra'] = regs.str.contains('Digital Registra', na=False).astype(int)
    feat_dict['reg_cloud_hosting'] = regs.str.contains('Cloud Hosting', na=False).astype(int)
    feat_dict['reg_neva'] = regs.str.contains('Neva', na=False).astype(int)
    feat_dict['reg_pandi_net'] = regs.str.contains('PANDI Registrar Net', na=False).astype(int)
    feat_dict['reg_cyberindo'] = regs.str.contains('Cyberindo', na=False).astype(int)
    feat_dict['reg_jagoan'] = regs.str.contains('Jagoan', na=False).astype(int)
    feat_dict['reg_aksara'] = regs.str.contains('Aksara', na=False).astype(int)
    feat_dict['reg_biznet'] = regs.str.contains('Biznet', na=False).astype(int)
    feat_dict['reg_webmedia'] = regs.str.contains('Web Media', na=False).astype(int)

    disc_dt = pd.to_datetime(df['discovered'], format='%m/%d/%Y %H:%M', errors='coerce')
    reg_dt = pd.to_datetime(df['registration_date'], errors='coerce')
    feat_dict['disc_month'] = disc_dt.dt.month.fillna(-1).astype(int)
    feat_dict['disc_year'] = disc_dt.dt.year.fillna(-1).astype(int)
    feat_dict['disc_hour'] = disc_dt.dt.hour.fillna(-1).astype(int)
    feat_dict['disc_day'] = disc_dt.dt.day.fillna(-1).astype(int)
    feat_dict['is_aug24'] = ((feat_dict['disc_year'] == 2024) & (feat_dict['disc_month'] == 8)).astype(int)
    feat_dict['is_aug2_2024'] = ((feat_dict['disc_year'] == 2024) & (feat_dict['disc_month'] == 8) & (feat_dict['disc_day'] == 2)).astype(int)
    feat_dict['is_late_apr24'] = ((feat_dict['disc_year'] == 2024) & (feat_dict['disc_month'] == 4) & (feat_dict['disc_day'] >= 27)).astype(int)
    feat_dict['is_q1_2024'] = ((feat_dict['disc_year'] == 2024) & (feat_dict['disc_month'].isin([1, 2, 3]))).astype(int)
    domain_age = (disc_dt - reg_dt).dt.days.fillna(-999)
    feat_dict['domain_age_days'] = domain_age
    feat_dict['future_registration'] = (domain_age < 0).astype(int)
    feat_dict['legacy_domain'] = (reg_dt.dt.year < 2010).astype(int)

    feat_dict['aug2_jagat_id_bare'] = (feat_dict['is_aug2_2024'] & feat_dict['reg_jagat'] & feat_dict['sld_id'] & feat_dict['url_no_scheme']).astype(int)
    feat_dict['q1_24_root_nobrand'] = (feat_dict['is_q1_2024'] & feat_dict['is_root_domain'] & feat_dict['brand_is_missing'] & feat_dict['conf_is_100']).astype(int)
    feat_dict['apr24_root_nobrand'] = (feat_dict['is_late_apr24'] & feat_dict['is_root_domain'] & feat_dict['brand_is_missing']).astype(int)
    feat_dict['edu_library_malware'] = ((feat_dict['sld_ac_id'] | feat_dict['sld_sch_id'] | feat_dict['sld_go_id']) & (feat_dict['ext_repo'] | feat_dict['path_slims']) & (feat_dict['kw_gambling_total'] == 0)).astype(int)
    feat_dict['spam_bare_co_or_net'] = (feat_dict['url_no_scheme'] & (feat_dict['sld_co_id'] | feat_dict['sld_net_id'])).astype(int)

    return pd.DataFrame(feat_dict, index=df.index).fillna(0)

def make_text_corpus(df):
    urls = df['url'].fillna('').astype(str)
    brands = df['brand'].fillna('').astype(str).replace({'-': '', 'nan': '', 'None': ''})
    slds = df['sld'].fillna('').astype(str)
    regs = df['registrar'].fillna('').astype(str)
    return urls + ' ' + brands + ' ' + slds + ' ' + regs

print('[*] Extracting tabular features and text corpus...')
X_test_feat = extract_features(test_df)
X_test_text = make_text_corpus(test_df)

# Align feature columns strictly with model
feature_cols = list(xgb_models[0].feature_names_in_)
X_test_feat = X_test_feat[feature_cols]
print(f'[OK] Features extracted: {X_test_feat.shape[1]} tabular columns for {X_test_feat.shape[0]} test samples.')

[*] Extracting tabular features and text corpus...


[OK] Features extracted: 272 tabular columns for 1500 test samples.


In [7]:
# CELL 7: Quad-Branch Inference Across 5 Folds
print('[*] Computing out-of-sample predictions across 5 folds for 4 branches...')

N_SAMPLES = len(test_df)
N_CLASSES = len(classes)

xgb_proba = np.zeros((N_SAMPLES, N_CLASSES))
lgb_proba = np.zeros((N_SAMPLES, N_CLASSES))
cat_proba = np.zeros((N_SAMPLES, N_CLASSES))
nlp_proba = np.zeros((N_SAMPLES, N_CLASSES))

for fold in range(5):
    print(f'  --> Running Fold {fold + 1}/5 inference...')
    # XGBoost
    xgb_proba += xgb_models[fold].predict_proba(X_test_feat) / 5.0
    # LightGBM
    lgb_proba += lgb_models[fold].predict_proba(X_test_feat) / 5.0
    # CatBoost
    cat_proba += cat_models[fold].predict_proba(X_test_feat) / 5.0
    # NLP LinearSVC (Calibrated)
    tfidf_pipe, nlp_model = nlp_models[fold]
    nlp_proba += nlp_model.predict_proba(tfidf_pipe.transform(X_test_text)) / 5.0

print('[OK] Inference completed for all 20 base models.')

[*] Computing out-of-sample predictions across 5 folds for 4 branches...
  --> Running Fold 1/5 inference...


  --> Running Fold 2/5 inference...


  --> Running Fold 3/5 inference...


  --> Running Fold 4/5 inference...


  --> Running Fold 5/5 inference...


[OK] Inference completed for all 20 base models.


In [8]:
# CELL 8: Optimal Quad-Branch Probability Blending
w_lgb, w_cat, w_xgb, w_nlp = opt_weights
print(f'[*] Applying Nelder-Mead ensemble weights:')
print(f'    LGB : {w_lgb:.4f}')
print(f'    CAT : {w_cat:.4f}')
print(f'    XGB : {w_xgb:.4f}')
print(f'    NLP : {w_nlp:.4f}')

blended_proba = (w_lgb * lgb_proba +
                 w_cat * cat_proba +
                 w_xgb * xgb_proba +
                 w_nlp * nlp_proba)

print(f'[OK] Blended probability matrix shape: {blended_proba.shape}')

[*] Applying Nelder-Mead ensemble weights:
    LGB : 0.2874
    CAT : 0.2734
    XGB : 0.2734
    NLP : 0.1658
[OK] Blended probability matrix shape: (1500, 7)


In [9]:
# CELL 9: Threshold Calibration & Calibrated Domain Threat Routing
def apply_domain_threat_routing(df, probs, alphas):
    scaled_probs = probs * alphas
    raw_preds = scaled_probs.argmax(axis=1)
    pred_labels = [idx_to_cls[p] for p in raw_preds]
    
    df_clean = df.reset_index(drop=True)
    overrides = 0
    
    for i in range(len(df_clean)):
        url_raw   = str(df_clean.at[i, 'url'])
        url_lower = url_raw.lower()
        brand_raw = str(df_clean.at[i, 'brand']).strip()
        sld_raw   = str(df_clean.at[i, 'sld'])
        disc_raw  = str(df_clean.at[i, 'discovered'])
        conf_val  = pd.to_numeric(df_clean.at[i, 'confidence_level'], errors='coerce')
        
        clean_url = url_raw.replace('http://', '').replace('https://', '')
        has_path   = '/' in clean_url and len(clean_url.split('/', 1)[1].strip('/')) > 0
        has_scheme = url_raw.startswith('http://') or url_raw.startswith('https://')
        
        dt = None
        for fmt in ('%m/%d/%Y %H:%M', '%Y-%m-%d %H:%M:%S', '%m/%d/%Y'):
            try:
                dt = datetime.strptime(disc_raw, fmt)
                break
            except:
                pass
                
        # Rule 1: High-confidence phishing targets (99%+ precision in IDADX)
        PURE_PHISH_BRANDS = {
            'DANA', 'Facebook', 'Facebook, Inc.', 'Telegram', 'WhatsApp',
            'Bank Permata', 'Permata', 'Bank BRI', 'BRI', 'Coda Payments',
            'Tencent', 'Garena', 'Meta', 'TikTok', 'Apple', 'BCA', 'BNI'
        }
        if brand_raw in PURE_PHISH_BRANDS:
            if pred_labels[i] != 'phishing':
                pred_labels[i] = 'phishing'
                overrides += 1
                continue
                
        # Rule 2: Confidence level == 90 (99.5% Phishing in IDADX)
        if conf_val == 90:
            if pred_labels[i] != 'phishing':
                pred_labels[i] = 'phishing'
                overrides += 1
                continue
                
        # Rule 3: Educational Library / SLiMS / Repository Malware
        is_edu_gov = sld_raw in ['ac.id', 'sch.id', 'go.id']
        has_repo = bool(re.search(r'repository|slims|pustaka|perpustakaan|digilib|\.exe|\.apk|osminogs', url_lower))
        has_gambling = any(kw in url_lower for kw in ['slot', 'judi', 'gacor', 'togel', 'cuan', 'zeus', 'maxwin', 'bo177'])
        if is_edu_gov and has_repo and not has_gambling:
            if pred_labels[i] != 'malware':
                pred_labels[i] = 'malware'
                overrides += 1
                continue
                
        # Rule 4: Active FakeShop wave detection in late April 2024
        if dt and dt.year == 2024 and dt.month == 4 and dt.day >= 26:
            if not has_path and (pd.isna(df_clean.at[i, 'brand']) or brand_raw in ('', '-', 'nan')):
                if pred_labels[i] in ('brand', 'other'):
                    pred_labels[i] = 'fakeshop'
                    overrides += 1
                    continue
                    
        # Rule 5: Brand Cybersquatting Dispute in Q1 2024
        if dt and dt.year == 2024 and dt.month in [1, 2, 3]:
            if not has_path and (pd.isna(df_clean.at[i, 'brand']) or brand_raw in ('', '-', 'nan')):
                if url_raw.startswith('http://') and pred_labels[i] == 'other':
                    brand_idx = cls_to_idx.get('brand', -1)
                    if brand_idx != -1 and probs[i, brand_idx] > 0.12:
                        pred_labels[i] = 'brand'
                        overrides += 1
                        continue
                        
        # Rule 6: August 2, 2024 Bare Domain Other
        if dt and dt.year == 2024 and dt.month == 8 and dt.day == 2:
            if not has_scheme and sld_raw == 'id' and (pd.isna(df_clean.at[i, 'brand']) or brand_raw in ('', '-', 'nan')):
                if pred_labels[i] == 'online gambling':
                    other_idx = cls_to_idx.get('other', -1)
                    if other_idx != -1 and probs[i, other_idx] > 0.15:
                        pred_labels[i] = 'other'
                        overrides += 1
                        continue
                        
        # Rule 7: Infrastructure Spam on bare co.id / net.id
        if not has_scheme and sld_raw in ['net.id', 'co.id']:
            if pred_labels[i] in ['online gambling', 'phishing']:
                pred_labels[i] = 'spam'
                overrides += 1
                continue
                
    print(f'[*] Calibrated domain routing adjustments applied: {overrides}')
    return pred_labels

final_predictions = apply_domain_threat_routing(test_df, blended_proba, best_alphas)

print('\n[*] Final Calibrated Prediction Distribution:')
dist = Counter(final_predictions)
for label, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'    {label:20s}: {count:5d} ({count/len(final_predictions)*100:5.2f}%)')

[*] Calibrated domain routing adjustments applied: 2

[*] Final Calibrated Prediction Distribution:
    online gambling     :   984 (65.60%)
    phishing            :   402 (26.80%)
    other               :    48 ( 3.20%)
    spam                :    33 ( 2.20%)
    malware             :    24 ( 1.60%)
    brand               :     8 ( 0.53%)
    fakeshop            :     1 ( 0.07%)


In [10]:
# CELL 10: Generate and Verify Submission Output
print(f'[*] Exporting predictions to: {OUTPUT_PATH}')

# Template columns: id, category
submission = pd.DataFrame({
    'id': test_df['id'],
    'category': final_predictions
})

# Strict integrity assertions
assert len(submission) == len(test_df), f'Row count mismatch! Expected {len(test_df)}, got {len(submission)}'
assert submission['id'].notna().all(), 'Found NaN in ID column!'
assert submission['category'].notna().all(), 'Found NaN in category column!'
assert list(submission.columns) == ['id', 'category'], f'Columns mismatch: {list(submission.columns)}'

# Write primary CSV
submission.to_csv(OUTPUT_PATH, index=False)

# Write alternate CSV (submission_output.csv) for evaluator compatibility
submission.to_csv(OUTPUT_ALT_PATH, index=False)

print(f'[OK] Submission file successfully saved!')
print(f'     Primary   : {OUTPUT_PATH}')
print(f'     Alternate : {OUTPUT_ALT_PATH}')
print(f'     Total rows: {len(submission)}')
print(f'     Columns   : {list(submission.columns)}')

print('\nPreview (First 10 records):')
print(submission.head(10).to_string(index=False))


[*] Exporting predictions to: C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\submission_datascape_ReproduksiPenyisihan.csv
[OK] Submission file successfully saved!
     Primary   : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\submission_datascape_ReproduksiPenyisihan.csv
     Alternate : C:\ZAKY\lomba\pedas2026\Pedas-Final\pedas2026-reproduksi-penyisihan\submission_output.csv
     Total rows: 1500
     Columns   : ['id', 'category']

Preview (First 10 records):
                id        category
PEDAS-bf8bc60352a3        phishing
PEDAS-7dd13e6859b0 online gambling
PEDAS-0fe0d926e0d2        phishing
PEDAS-37db191c480d        phishing
PEDAS-2d06b168bfcc online gambling
PEDAS-60cb1dcf36fd online gambling
PEDAS-21d4a69584c6 online gambling
PEDAS-344e9c62f21e online gambling
PEDAS-54a6d1898c81 online gambling
PEDAS-cbb9cd0dbc6a online gambling
